# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a step-by-step exploration of the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. You will learn how to load Croissant metadata, inspect available record sets and fields, and perform exploratory data analysis using references by `@id` for all dataset entities.

### Dataset Source
FAIR² dataset Croissant schema: https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and instantiate the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL of the Croissant metadata schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# `dataset.metadata` is an object with dataset properties
print(f"Dataset title: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview

List all record sets in the dataset, including their `@id` and contained fields (by `@id`).

We'll use `dataset.record_sets` and for each, show its fields and columns.

In [ ]:
# List all record sets, their @id, and their fields
record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for i, rs in enumerate(record_sets):
    print(f"Record Set {i+1}:")
    print(f"  @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else '<no name>'}")
    # List fields
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for fld in rs.fields:
            print(f"    - @id: {fld.id}, name: {fld.name if hasattr(fld, 'name') else '<no name>'}")
    # List columns
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    - @id: {col.id}, name: {col.name if hasattr(col, 'name') else '<no name>'}")
    print()

## 3. Data Extraction

Identify record sets for tabular data. For demonstration, extract all tabular record sets into pandas DataFrames. Record sets, fields, and columns are referenced by their `@id`.

We'll pick the first tabular record set as an example for EDA.

In [ ]:
# Identify record set @ids
tabular_record_sets = [rs for rs in dataset.record_sets if getattr(rs, 'columns', None)]
tabular_record_set_ids = [rs.id for rs in tabular_record_sets]

print(f"Tabular Record Sets @id:")
for rid in tabular_record_set_ids:
    print(f"- {rid}")

# Extract all records from each tabular record set into a DataFrame, referenced by @id
dataframes = {}
for rs_id in tabular_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[rs_id] = df

# Print columns in the first tabular record set
if tabular_record_set_ids:
    first_rs_id = tabular_record_set_ids[0]
    print(f"\nColumns (by field @id) in first tabular record set '{first_rs_id}':\n", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Process a numeric field (by its @id): filter out outliers, normalize, and analyze grouped statistics. All field references use their Croissant `@id`.

**Replace field @ids below with those found in your overview.**

In [ ]:
# --- Example: EDA on first tabular record set ---
# Replace with actual @ids from your dataset overview as needed.
record_set_id = tabular_record_set_ids[0] if tabular_record_set_ids else None
df = dataframes[record_set_id]

# For demonstration, pick the first numeric-looking column (usually coefficients or statistics)
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is not None:
    print(f"Using numeric field '@id': {numeric_field_id}")
    # Example threshold for filtering
    threshold = df[numeric_field_id].mean() + df[numeric_field_id].std() if not df[numeric_field_id].isnull().all() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by the first non-numeric field (e.g., a factor or class/variable column)
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
            group_field_id = col
            break

    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA in this record set.")

## 5. Visualization

Visualize numeric field distributions and group comparisons (using `matplotlib`).

**Adjust field `@id` as needed for your dataset.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- We loaded dataset metadata via Croissant and identified available record sets and fields using their `@id`s.
- Example data extraction, filtering, normalization, and grouping were demonstrated using the selected tabular record set and fields by id.
- Visualization provides a quick overview of numeric distributions and group effects for regression outputs.

Explore further using more Croissant entities and visit the [`mlcroissant` documentation](https://github.com/mlcommons/croissant) for advanced options.